# Birr Track — Fine-tune Qwen2.5-VL-3B on receipts

This Kaggle notebook fine-tunes [Qwen/Qwen2.5-VL-3B-Instruct](https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct) on the Birr Track receipt dataset using LoRA via [LLaMA-Factory](https://github.com/hiyouga/LLaMA-Factory).

## Prerequisites

1. Run `python prepare_dataset.py` locally first. It produces `train.json`, `val.json`, and `dataset_info.json`.
2. Create a **Kaggle Dataset** containing:
   - The `documents/receipts/` folder (all receipt images, flat).
   - The generated `train.json` and `val.json`.
3. Attach that Kaggle Dataset to this notebook (right panel → **Add Data**).
4. In the right panel, set **Accelerator** to **GPU T4 x2** and **Internet** to **ON**.
5. Run all cells. The final cell writes a downloadable zip of the LoRA adapter.

## What this notebook does

- Installs LLaMA-Factory (pinned to a recent commit) plus Qwen2.5-VL utilities.
- Registers the prepared dataset with LLaMA-Factory's data registry.
- Trains a LoRA adapter (rank 16, alpha 32) over the language and projector layers, keeping the vision tower frozen.
- Runs a quick inference check on a held-out validation example.
- Zips the trained adapter to `/kaggle/working/qwen25vl-3b-birrtrack-lora.zip`.

In [ ]:
import subprocess

print(subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"]).decode())
print(subprocess.check_output(["python", "--version"]).decode())

## 1. Install LLaMA-Factory and Qwen2.5-VL dependencies

We clone `v0.9.2` and `pip install ".../LLaMA-Factory[torch,metrics]"` (avoids pip's rejected `#egg=pkg[extras]` URLs).

**Important:** Do not run a follow-up line that forces `peft==0.13.x` or other versions outside LLaMA-Factory's declared ranges. v0.9.2 requires `peft<=0.12.0` and `trl<=0.9.6`; overriding them triggers resolver warnings or broken training.

### Pip “dependency conflicts” / `ERROR` lines on Kaggle (usually safe to ignore)

Kaggle’s image ships **many** preinstalled packages (JAX, Colab helpers, OpenCV, etc.). LLaMA-Factory’s install may pull **NumPy 1.26.x** (and an older `fsspec`) to stay compatible with its PyTorch / training stack. Pip then prints long **“ERROR: … dependency conflicts”** messages: that is the **resolver warning**, not a failed build, as long as the cell finishes and LLaMA-Factory’s wheel built successfully.

For this notebook you only need **Torch + Transformers + LLaMA-Factory + Qwen helpers**; you are not using those other Kaggle libraries in the same workflow. **Do not** blindly `pip install -U numpy` to “fix” the warnings — that can break the training stack. The next cell runs a quick import check; if that passes, continue.

Only `qwen-vl-utils` is installed separately (image helpers for Qwen2.5-VL).


In [ ]:
!pip install -q --upgrade pip setuptools wheel
!pip install -q --upgrade "setuptools>=70"
!rm -rf /kaggle/working/LLaMA-Factory && git clone --depth 1 --branch v0.9.2 https://github.com/hiyouga/LLaMA-Factory.git /kaggle/working/LLaMA-Factory && pip install -q "/kaggle/working/LLaMA-Factory[torch,metrics]"
# LLaMA-Factory v0.9.2's requirements.txt pins `numpy<2.0.0`, so the install above
# downgrades numpy from Kaggle's preinstalled 2.x to 1.26.x. But Kaggle's
# scipy / scikit-learn / opencv / jax / cupy wheels are all compiled against numpy 2.x,
# and recent scipy versions (>=1.13) vendor an `array_api_compat` that references
# `numpy.strings` (a numpy 2.0+ symbol). After the LF downgrade, importing transformers
# trips one of:
#   ValueError: numpy.dtype size changed, ... Expected 96 ..., got 88
#   ModuleNotFoundError: No module named 'numpy.strings'
# The fix is to override LF's overly-conservative pin and put numpy back to 2.x — LF v0.9.2
# and all the deps we actually use (transformers 4.49, peft 0.12, datasets <=3.2, librosa,
# av, accelerate) work fine with numpy 2.x. Pip will print a red ERROR line about
# llamafactory wanting numpy<2 — ignore it, same as the other resolver warnings above.
#
# IMPORTANT: do NOT use loose `numpy>=2` only — pip can leave a *mixed* numpy tree (some
# files from an older wheel, some from newer), which then crashes scipy/transformers with:
#   ImportError: cannot import name '_center' from 'numpy._core.umath'
# Uninstall first, then force-reinstall *pinned* wheels that match each other.
!pip uninstall -y numpy scipy scikit-learn 2>/dev/null || true
!pip install -q --no-cache-dir --force-reinstall "numpy==2.2.2" "scipy==1.15.2" "scikit-learn==1.6.1"
# Pin transformers to the highest version allowed by LLaMA-Factory v0.9.2
# (range: >=4.41.2,<=4.49.0,!=4.46.*,!=4.47.*,!=4.48.0). Qwen2.5-VL's model classes
# (Qwen2_5_VLForConditionalGeneration) only ship in transformers >= 4.49.0, but pip's
# resolver leaves a preinstalled 4.41-4.45 alone since it already satisfies LF's range —
# which then breaks the inference cell with `cannot import name 'Qwen2_5_VLForConditionalGeneration'`.
!pip install -q --upgrade "transformers==4.49.0"
# Do not pin peft/accelerate/trl here — LLaMA-Factory v0.9.2 declares compatible ranges (e.g. peft<=0.12.0). Overriding those breaks the resolver.
!pip install -q qwen-vl-utils==0.0.8
# Debian/Kaggle base image ships an old pkg_resources; Python 3.12 removed pkgutil.ImpImporter.
# LLaMA-Factory imports jieba → pkg_resources → crash unless setuptools is fresh (see Train section notes).
!pip install -q --upgrade "setuptools>=70"

# Sanity check: training stack loads (ignore unrelated Kaggle image packages pip warned about).
import importlib

for _name in ("numpy", "torch", "transformers", "peft"):
    _m = importlib.import_module(_name)
    print(_name, getattr(_m, "__version__", "?"))
print("llamafactory", importlib.metadata.version("llamafactory"))

# Hard-fail early if transformers somehow ended up below 4.49 — saves a 5-minute
# train run that would die at the inference step.
import transformers as _tf

assert _tf.__version__ >= "4.49.0", (
    f"Qwen2.5-VL needs transformers>=4.49.0, got {_tf.__version__}. "
    "Re-run the cell or set Kaggle Language to Python 3.10."
)
from transformers import Qwen2_5_VLForConditionalGeneration  # noqa: F401  # smoke import

print("Qwen2.5-VL classes import OK.")


## 2. Configure dataset paths

Update `KAGGLE_DATASET_NAME` below to match the slug of the Kaggle Dataset you attached. The dataset is mounted read-only at `/kaggle/input/<slug>/`. The expected layout inside the dataset is:

```
<slug>/
├── train.json
├── val.json
└── receipts/
    ├── cbe_screenshot_1.jpg
    ├── photo_1@18-04-2026_10-57-45.jpg
    └── ...
```

The cell below also rewrites the `images` paths inside `train.json` / `val.json` to point at the mounted Kaggle dataset (since `prepare_dataset.py` writes absolute paths from the dev machine).

In [ ]:
import json
import os
from pathlib import Path

KAGGLE_DATASET_NAME = "birrtrack-receipts"
KAGGLE_DATASET_ROOT = Path(f"/kaggle/input/{KAGGLE_DATASET_NAME}")

if not KAGGLE_DATASET_ROOT.is_dir():
    candidates = sorted(Path("/kaggle/input").glob("*"))
    raise FileNotFoundError(
        f"Kaggle dataset not found at {KAGGLE_DATASET_ROOT}. "
        f"Update KAGGLE_DATASET_NAME above. Available datasets: {[c.name for c in candidates]}"
    )

receipts_dir = KAGGLE_DATASET_ROOT / "receipts"
train_src = KAGGLE_DATASET_ROOT / "train.json"
val_src = KAGGLE_DATASET_ROOT / "val.json"

for required in (receipts_dir, train_src, val_src):
    if not required.exists():
        raise FileNotFoundError(f"Missing required path inside the Kaggle dataset: {required}")

work_data_dir = Path("/kaggle/working/birrtrack_data")
work_data_dir.mkdir(parents=True, exist_ok=True)


def rewrite_image_paths(src_json: Path, dst_json: Path) -> tuple[int, int]:
    """Rewrite per-example image paths to point at `receipts_dir`.

    Returns (num_examples, num_images_rewritten) so the caller can sanity-check
    the dataset shape after the rewrite. Also fails loudly if any referenced
    image is missing from the Kaggle dataset (catches typos before training).
    """
    examples = json.loads(src_json.read_text(encoding="utf-8"))
    rewritten = 0
    missing: list[str] = []
    for ex in examples:
        new_images = []
        for img in ex.get("images", []):
            file_name = os.path.basename(img)
            new_path = receipts_dir / file_name
            if not new_path.is_file():
                missing.append(file_name)
            new_images.append(str(new_path))
            rewritten += 1
        ex["images"] = new_images
    if missing:
        sample = ", ".join(missing[:5])
        raise FileNotFoundError(
            f"{len(missing)} image(s) referenced in {src_json.name} are missing from "
            f"{receipts_dir}. First few: {sample}"
        )
    dst_json.write_text(json.dumps(examples, ensure_ascii=False, indent=2), encoding="utf-8")
    return len(examples), rewritten


train_dst = work_data_dir / "train.json"
val_dst = work_data_dir / "val.json"

train_count, train_imgs = rewrite_image_paths(train_src, train_dst)
val_count, val_imgs = rewrite_image_paths(val_src, val_dst)

print(f"Train: {train_count} examples, {train_imgs} images")
print(f"Val:   {val_count} examples, {val_imgs} images")

## 3. Register the dataset with LLaMA-Factory

LLaMA-Factory reads dataset definitions from a `dataset_info.json` file in its data directory. We write a minimal one that points at our prepared JSON files.

In [ ]:
dataset_info = {
    "birrtrack_train": {
        "file_name": str(train_dst),
        "formatting": "sharegpt",
        "columns": {"messages": "conversations", "images": "images"},
        "tags": {
            "role_tag": "from",
            "content_tag": "value",
            "user_tag": "human",
            "assistant_tag": "gpt",
        },
    },
    "birrtrack_val": {
        "file_name": str(val_dst),
        "formatting": "sharegpt",
        "columns": {"messages": "conversations", "images": "images"},
        "tags": {
            "role_tag": "from",
            "content_tag": "value",
            "user_tag": "human",
            "assistant_tag": "gpt",
        },
    },
}

dataset_info_path = work_data_dir / "dataset_info.json"
dataset_info_path.write_text(json.dumps(dataset_info, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Wrote {dataset_info_path}")

## 4. Training config

The values below are tuned for Kaggle's free **T4 x2** (16 GB per GPU). They run single-GPU by default for predictability. To use both GPUs, switch the launch command from `llamafactory-cli train` to `torchrun --nproc_per_node=2 ...` (commented out below).

Key choices:

- LoRA rank 16, alpha 32, applied to all linear modules.
- Vision tower frozen (`freeze_vision_tower: true`) — saves memory and the ViT generalizes well enough out of the box.
- fp16 (T4 does not support bf16).
- Gradient checkpointing + accum 8 to fit batch_size=1 in 16 GB.
- 10 epochs on ~90 examples is short (~3-6 minutes on T4) — enough for a small structured-extraction task.

In [ ]:
import yaml

OUTPUT_DIR = Path("/kaggle/working/qwen25vl-3b-birrtrack-lora")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_config = {
    "model_name_or_path": "Qwen/Qwen2.5-VL-3B-Instruct",
    "trust_remote_code": True,

    "stage": "sft",
    "do_train": True,
    "finetuning_type": "lora",
    "lora_target": "all",
    "lora_rank": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "freeze_vision_tower": True,

    "dataset": "birrtrack_train",
    "eval_dataset": "birrtrack_val",
    "dataset_dir": str(work_data_dir),
    "template": "qwen2_vl",
    "cutoff_len": 2048,
    "overwrite_cache": True,
    "preprocessing_num_workers": 2,

    "output_dir": str(OUTPUT_DIR),
    "overwrite_output_dir": True,
    "logging_steps": 5,
    "save_steps": 50,
    "save_total_limit": 2,
    "plot_loss": True,

    "per_device_train_batch_size": 1,
    "per_device_eval_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "learning_rate": 1.0e-4,
    "num_train_epochs": 10.0,
    "lr_scheduler_type": "cosine",
    "warmup_ratio": 0.03,
    "fp16": True,
    "gradient_checkpointing": True,

    "eval_strategy": "epoch",
    "load_best_model_at_end": False,

    "report_to": "none",
    "seed": 42,
}

config_path = Path("/kaggle/working/train_config.yaml")
config_path.write_text(yaml.safe_dump(train_config, sort_keys=False), encoding="utf-8")
print(f"Wrote {config_path}")
print(config_path.read_text())

## 5. Train

### Harmless log noise (you can ignore)

Lines like `Unable to register cuFFT/cuDNN/cuBLAS factory` and XLA `computation placer already registered` come from **TensorFlow/XLA and PyTorch both touching CUDA** on the same kernel. They do not stop training.

### Python 3.12 + `ImpImporter` crash (fatal — fixed below)

If `llamafactory-cli` dies with `AttributeError: module 'pkgutil' has no attribute 'ImpImporter'` inside `jieba` → `pkg_resources`, the Kaggle image is loading an **old** `pkg_resources` from `/usr/lib/python3/dist-packages/`. Python 3.12 removed `ImpImporter`; **setuptools ≥ 70** ships a compatible `pkg_resources`.

The install cell upgrades setuptools twice (before and after LLaMA-Factory). The train cell upgrades setuptools once more immediately before launch so nothing in between can pin an old version.

**Fallback:** In the notebook right panel, set **Language** to **Python 3.10** if your runtime still picks a broken system `pkg_resources`.

### Run training

Expect ~5–15 minutes for 10 epochs on a T4 with this dataset size. Watch `loss` trend downward; if it stays flat at a high value, re-check dataset registration in step 3.


In [ ]:
import os
import subprocess
import sys
import sysconfig

# setuptools>=70 bundles pkg_resources that works on Python 3.12. Kaggle still ships an old
# /usr/lib/.../pkg_resources that crashes (ImpImporter). Force-reinstall, then prepend pip's
# platlib path via PYTHONPATH so the training subprocess shadows the Debian copy.
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "--force-reinstall", "setuptools>=70"]
)
_local_site_packages = sysconfig.get_path("platlib")
env = os.environ.copy()
# Pin to a single GPU so LLaMA-Factory's CLI does NOT auto-fork torchrun
# (which it does whenever it sees >1 visible device — see llamafactory/cli.py TRAIN branch).
env["CUDA_VISIBLE_DEVICES"] = "0"
# Quiet TF/XLA cuFFT/cuDNN/cuBLAS "Unable to register factory" noise.
env.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
# Kaggle ships wandb. Without these, transformers Trainer can pop login prompts / send telemetry.
env["WANDB_DISABLED"] = "true"
env["WANDB_MODE"] = "disabled"
env["HF_HUB_DISABLE_TELEMETRY"] = "1"
env["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
# Single-process training — turn off the tokenizers fork warning.
env.setdefault("TOKENIZERS_PARALLELISM", "false")

_pp_existing = env.get("PYTHONPATH", "").strip()
if _local_site_packages:
    # Local pip site-packages must come *first* in PYTHONPATH so we shadow /usr/lib's broken pkg_resources.
    env["PYTHONPATH"] = (
        _local_site_packages
        + ((os.pathsep + _pp_existing) if _pp_existing else "")
    )

try:
    subprocess.check_call(
        [sys.executable, "-m", "llamafactory.cli", "train", "/kaggle/working/train_config.yaml"],
        env=env,
    )
except subprocess.CalledProcessError as exc:
    raise RuntimeError(
        f"llamafactory-cli train exited with code {exc.returncode}. Scroll up for the real "
        "stack trace (usually the last few lines before this message)."
    ) from exc

# LLaMA-Factory writes the final adapter to OUTPUT_DIR at end of training.
# Fail fast here if it didn't, instead of letting the inference cell raise a confusing peft error.
_adapter_cfg = OUTPUT_DIR / "adapter_config.json"
_adapter_weights = OUTPUT_DIR / "adapter_model.safetensors"
if not _adapter_cfg.is_file() or not _adapter_weights.is_file():
    raise FileNotFoundError(
        f"Training finished but no LoRA adapter was saved to {OUTPUT_DIR}. "
        f"Expected adapter_config.json + adapter_model.safetensors. "
        f"Contents: {sorted(p.name for p in OUTPUT_DIR.iterdir())}"
    )
print(f"Adapter saved at: {OUTPUT_DIR}")

# To use both T4s, comment out the block above and use:
# env["CUDA_VISIBLE_DEVICES"] is unset or "0,1"
# subprocess.check_call(
#     [
#         sys.executable, "-m", "torch.distributed.run",
#         "--nproc_per_node", "2", "--master_port", "29500",
#         "-m", "llamafactory.cli", "train", "/kaggle/working/train_config.yaml",
#     ],
#     env=env,
# )


## 6. Sanity-check inference on one validation image

Load the base model + the freshly trained LoRA adapter and run a single prediction against the first validation example. The output should be a JSON string matching the label schema. This catches issues like wrong chat template, missing decoder tokens, or empty adapter files before you spend time evaluating.

In [ ]:
import gc

import torch
from peft import PeftModel
from qwen_vl_utils import process_vision_info
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

BASE_MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
INFERENCE_DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

# Reclaim any GPU memory the kernel was still holding from earlier cells before we
# load a fresh ~6GB fp16 model. Without this, repeated notebook runs can OOM on T4.
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

processor = AutoProcessor.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
# Pin everything to cuda:0 explicitly. `device_map="auto"` on a Kaggle T4 x2 kernel can
# silently split the model across both GPUs (since both are visible here — only the
# training subprocess restricted itself via CUDA_VISIBLE_DEVICES). A split model breaks
# `inputs.to(model.device)` and `model.generate` for VLMs.
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float16,
    device_map={"": INFERENCE_DEVICE},
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
model = PeftModel.from_pretrained(base_model, str(OUTPUT_DIR))
model.eval()

# Qwen2.5-VL has no real pad token; transformers warns and falls back to eos at generate-time.
# Set it explicitly so generate() doesn't print the warning every call.
pad_token_id = processor.tokenizer.pad_token_id or processor.tokenizer.eos_token_id

val_examples = json.loads(val_dst.read_text(encoding="utf-8"))
sample = val_examples[0]
sample_image_path = sample["images"][0]
sample_label = sample["conversations"][1]["value"]

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": sample_image_path},
            {"type": "text", "text": sample["conversations"][0]["value"].replace("<image>", "").strip()},
        ],
    }
]

text_prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text_prompt],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
).to(INFERENCE_DEVICE)

with torch.inference_mode():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        pad_token_id=pad_token_id,
    )

# Move IDs to CPU before slicing/decoding. `batch_decode` works on CUDA tensors in
# modern transformers, but the slice-then-decode pattern is more portable on CPU.
input_lengths = inputs.input_ids.shape[1]
trimmed_ids = output_ids[:, input_lengths:].cpu()
generated = processor.batch_decode(trimmed_ids, skip_special_tokens=True)[0]

print(f"Image:    {sample_image_path}")
print(f"Expected: {sample_label}")
print(f"Got:      {generated.strip()}")

## 7. Package the LoRA adapter for download

The final zip lands in `/kaggle/working/` and is downloadable from the right-hand panel under **Output**.

In [ ]:
import os
import shutil

zip_path = shutil.make_archive(
    base_name="/kaggle/working/qwen25vl-3b-birrtrack-lora",
    format="zip",
    root_dir=str(OUTPUT_DIR),
)
print(f"Adapter zipped to {zip_path}")
print(f"Size (MB): {os.path.getsize(zip_path) / 1024 / 1024:.1f}")
